<h1 style="font-family: Roboto; width: 85%;">Computer Assignment #5</h1>
<h2>Intelligent Information Retrieval<br>
University of Tehran<br>
Fall 2024</h2>

<div style="font-family: Roboto; width: 85%; font-size: 18px;">
Name: Erfan Shahabi
<br/>


**نکات و توضیحات تکمیلی آخر فایل تمرین را به دقت مطالعه فرمایید**

# **بخش اول:**

In [1]:
!pip install datasets

<div style="font-family: Roboto; width: 85%; font-size: 18px;">
در این بخش از کد دیتا ست را وارد کردم و دیتا ست را به سه بخش ترین، تست و ارزیابی تقسیم کردم.
چون من برنامه را در محیط گوگل کولب نوشتم بعد از ایمپورت کردن کتابخانه های مورد نیاز ابتدا با استفاده از کتابخانه google.colab به محیط گوگل درایو متصل شدم و دیتا ست را از از گوگل درایو بارگذاری کردم. سپس ۸۰ درصد داده ها را به عنوان داده های آموزشی جدا کردم و از ۲۰ درصد باقی مانده نیمی را به عنوان داده های تست و نیمی را به عنوان داده های ارزیابی دسته بندی نمودم.

<div style="font-family: Roboto; width: 85%; font-size: 18px;">

In [2]:
#YOUR CODE HERE
from google.colab import drive
from datasets import load_dataset
drive.mount('/content/drive')
dataset = load_dataset('csv', data_files='/content/drive/MyDrive/IR5/dataset.csv')
train_test_val_split = dataset['train'].train_test_split(test_size=0.2)
train_data = train_test_val_split['train']
test_val_data = train_test_val_split['test']
test_val_split = test_val_data.train_test_split(test_size=0.5)
test_data = test_val_split["test"]
val_data = test_val_split["train"]
print("train size:", len(train_data))
print('test size:', len(test_data))
print('validation size:', len(val_data))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train size: 2676
test size: 335
validation size: 335


<div style="font-family: Roboto; width: 85%; font-size: 18px;">
در این قسمت ابتدا نمونه ای از داده ها را چاآن ها را نیز طبق خواسته سوال چاپ کردم. همانطور که از نتایج مشخص است می توان گفت برچسب ها به طور مناسب توزیع شده اند و در هر دسته از داده ها برچسب های ۰ و ۱ توزیعی نزدیک به هم دارند پس از نظر توزیع این دسته بندی مناسب عمل کرده است.

In [3]:
print('train samples:')
print(train_data[:2])
print('validation samples:')
print(val_data[:2])
print('test samples')
print(test_data[:2])
from collections import Counter

train_label_distribution = Counter(train_data['labels'])

val_label_distribution = Counter(val_data['labels'])

test_label_distribution = Counter(test_data['labels'])

print("Train Label Distribution:", train_label_distribution)
print("Validation Label Distribution:", val_label_distribution)
print("Test Label Distribution:", test_label_distribution)

train samples:
{'query': ['renewable power', 'renewable power'], 'document': ['A plan that would allow a private partnership to siphon off methane gas from \nthe Lopez Canyon Landfill could be ready to go to the City Council this month, \naccording to city officials. The proposed methane-energy plant was approved by the Los Angeles Board of \nPublic Works last September, and sent to the city administrative office for \nanalysis. The methane, a byproduct of decaying trash, is percolating through the soil of \nthe dump, which is slated to close in 1996. The malodorous gas, long the bane \nof area residents, is being collected and burned off. Lopez Canyon Energy \nPartners hopes to harness that wasted energy to generate electricity. But the city administrative office\'s approval of a lease agreement with the \ncity had to be put on a back burner because the office is tied up preparing the \ncity\'s overall budget, said Richard Hart, principal administrative analyst for \nthe office. "With

In [4]:
!pip install nltk

<div style="font-family: Roboto; width: 85%; font-size: 18px;">

In [5]:
def filter_valid_rows(example):
    return (
        isinstance(example["query"], str) and example["query"] != "" and
        isinstance(example["document"], str) and example["document"] != ""
    )

train_data = train_data.filter(filter_valid_rows)
val_data = val_data.filter(filter_valid_rows)
test_data = test_data.filter(filter_valid_rows)

print("Filtered train samples:", train_data[:2])
print("Filtered validation samples:", val_data[:2])
print("Filtered test samples:", test_data[:2])

Filter:   0%|          | 0/2676 [00:00<?, ? examples/s]

Filter:   0%|          | 0/335 [00:00<?, ? examples/s]

Filter:   0%|          | 0/335 [00:00<?, ? examples/s]

Filtered train samples: {'query': ['renewable power', 'russian intervention in chechnya'], 'document': ['A plan that would allow a private partnership to siphon off methane gas from \nthe Lopez Canyon Landfill could be ready to go to the City Council this month, \naccording to city officials. The proposed methane-energy plant was approved by the Los Angeles Board of \nPublic Works last September, and sent to the city administrative office for \nanalysis. The methane, a byproduct of decaying trash, is percolating through the soil of \nthe dump, which is slated to close in 1996. The malodorous gas, long the bane \nof area residents, is being collected and burned off. Lopez Canyon Energy \nPartners hopes to harness that wasted energy to generate electricity. But the city administrative office\'s approval of a lease agreement with the \ncity had to be put on a back burner because the office is tied up preparing the \ncity\'s overall budget, said Richard Hart, principal administrative analy

<div style="font-family: Roboto; width: 85%; font-size: 20px;">
در این قست از کد تلاش کردم تا مراحل پیش پردازش را روی داده ها انجام بدم و با استفاده مدل از پیش آموزش دیده و توکایزر آن داده ها را توکنایز کنم.
ابتدا کتابخانه های مورد نظر را وارد برنامه کرده ام و از کتابخانه nltk.corpus استاپ ورد های انگلسی را استخراج کردم. من برای مدل از نسخه پایه ای bert-base-uncased استفاده کزدم و توکنایزر مربوط به این مدل را نیز بارگذاری کردم و در مرحله بعد مدل طبقه بند را نیز بارگذاری کردم. این مدل وظیفه طبثه بندی باینری بین لیبل ها را بر عهده دارد.
سپس وزن های مدل پایه را فریز کردم  و تغییرات فقط در لایه آخر مدل اغمال می شوند.
بعد از آن استاپ ورد ها را آماده سازی کردم و مجموعه استاپ ورد های زبان انگلیسی را در متغیر stop_words ذخیره کردم.
در مرحله بعد تابع حذف استاپ ورد را تعریف کردم که این تابع  استاپ ورد ها را از داده ها حذف میکند.
سپس تابع پیش پردازش را تعریف کردم که  ابتدا بررسی میکند که هر کوئری یک رشته است با خیر و اگر نبود مقدار خالی "" را جایگزین می کند. سپس با استفاده از  تابع حذقف استاپ ورد، استاپ ورد ها را از داده ها حذف کردم و سپس با استفاده از توکنایزر وارد شده در کد جفت های کوئری و سند را توکنایز کردم.
همچین در این تابع برای یکسان سازی طول داده ها پدینگ نیز انجاک  دادم. و در نهایت پیش پردازش را روی مجموعه داده ها اعمال کردم.


In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk


nltk.download('stopwords')
nltk.download('punkt')

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


for param in model.base_model.parameters():
    param.requires_grad = False


stop_words = set(stopwords.words('english'))


def remove_stopwords(text, tokenizer, stop_words):
    tokens = tokenizer.tokenize(text)
    filtered_tokens = [token for token in tokens if token.lower() not in stop_words]
    return tokenizer.convert_tokens_to_string(filtered_tokens)

def preprocess_function(examples):

    queries_no_stopwords = [
        remove_stopwords(query, tokenizer, stop_words) if isinstance(query, str) else ""
        for query in examples["query"]
    ]
    documents_no_stopwords = [
        remove_stopwords(document, tokenizer, stop_words) if isinstance(document, str) else ""
        for document in examples["document"]
    ]

    return tokenizer(
        queries_no_stopwords,
        documents_no_stopwords,
        padding=True,
        truncation=True,
        max_length=128,
    )

tokenized_train_data = train_data.map(preprocess_function, batched=True)
tokenized_val_data = val_data.map(preprocess_function, batched=True)
tokenized_test_data = test_data.map(preprocess_function, batched=True)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able 

Map:   0%|          | 0/2644 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (740 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/331 [00:00<?, ? examples/s]

Map:   0%|          | 0/331 [00:00<?, ? examples/s]

In [7]:
pip install --upgrade evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.1 MB/s eta 0:00:00


<div style="font-family: Roboto; width: 85%; font-size: 20px;">
در این قسمت از تمرین سعی کردم کدی بنویسم که با استفاده از آن مدل فرایند آموزش را انجام دهد و ارزیابی های خواسته شده در سوال در حین اجرای کد نمایش داده شود.
بعد از ایمپورت کردن کتابخانه های مورد نیاز تابع محاسبه معیار های ارزیابی را نوشتم که این تابع خروجی مدل و برچسب آن را دریافت می کند و با استفاده از تابع softmax با توجه به بیشترین احتمال برای هر نمونه برچسب داده را پیش بینی می کند. همچنین این تابع معیاز های ارزیابی را هم نمایش می دهد. معیار accuracy نشان دهنده درصد پیش بینی های صحیح، precision نشان دهنده دقت پیش بینی برای کلاس مثبت است. recallنشان دهنده نسبت بازیابی کلاس مثبت است.
معیار F1 میانگین هماهنگ precision , recall و معیار ROC AUC هم مساحت زیر منحنی ROC است.
در این تابع در صورت وقوع خطا به دلیل تک کلاس بودن پیش بینی ها مقدار NaN بازگردادنده می شود.
در مرحله بعدی پارامتر های آموزش تعریف شده اند و بعد از آن ترینر را تغریف کرده ام سپس با استفاده از trainer.train() مدل در حالت آموزش و با استفاده از trainer.evaluate() مدل در حالت ارزیابی قرار می گیرد.
 و در پابان مقادیر محاسبه شده برای هر معیار نمایش داده می شود.
 این مقادیر به شررح زیر می باشند:
 <br>
eval_accuracy: 0.8606
eval_precision: 0.8098
eval_recall: 0.9313
eval_f1: 0.8663
eval_roc_auc: 0.9575
 <br>
مقدار accuracy برابر ۸۶٪ می باشد که این موضوع نشان می دهد حدودا ۸۶٪ پیش بینی ها درست بوده اند.
معیار precision برای این مدل برابر ۰.۰۸۰۹۸ نی باشد که نشان دهنده این است مدل توانسته حدودا در ۸۰ درصد مواقع سند مرتبط را درست تشخیص داده است همچنین معیار recall برای این مدل برابر ۹۳.۱۳ بووده است که نشان میدهد مدل توانشته ۹۳ درصد اسناد مرتبط را به درستی بازیابی کند. معیار f1 که میانگینی وزن دار بین دقت و قراخوانی است برابر ۰.۸۶۶۳ بوده که نشان گر تناسب بین دقت و فراخوانی در این مدل است.
همچین معیار roc aucکه مساحت زیر نمودار rocرا نشان می دهد برابر ۰.۹۵۷۵ بوده است.
همچنین با نوجه به مقادیر lossمنتشر شده در حین آموزش و ارزیابی مدل نشان می دهد مدل به خوبی فرایند آموزش را طی کرده و overfit نکرده است.

In [8]:
import evaluate
from sklearn.metrics import roc_auc_score
import numpy as np
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    probs = np.exp(logits) / np.sum(np.exp(logits), axis=-1, keepdims=True)


    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, zero_division=0)
    recall = recall_score(labels, predictions, zero_division=0)
    f1 = f1_score(labels, predictions, zero_division=0)

    try:
        auc = roc_auc_score(labels, probs[:, 1])
    except ValueError:
        auc = float('nan')

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": auc,
    }

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/IR5/bert_model_output",
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="/content/drive/MyDrive/IR5/logs",
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_data,
    eval_dataset=tokenized_val_data,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


trainer.train()

results = trainer.evaluate(eval_dataset=tokenized_val_data)

print("Evaluation Results:")
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-8-d787120a1f59>:46: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.433200,0.265990,0.878049,0.817259,0.975758,0.889503,0.981037
2,0.319000,0.230260,0.905488,0.864130,0.963636,0.911175,0.981260
3,0.297800,0.222750,0.911585,0.873626,0.963636,0.916427,0.981223


Evaluation Results:
eval_loss: 0.3094
eval_accuracy: 0.8606
eval_precision: 0.8098
eval_recall: 0.9313
eval_f1: 0.8663
eval_roc_auc: 0.9575
eval_runtime: 2.3217
eval_samples_per_second: 142.1360
eval_steps_per_second: 18.0900
epoch: 3.0000


##سوالات مفهومی
<div style="font-family: Roboto; width: 85%; font-size: 20px;">
الف)
مدل‌های ترنسفورمر مانند برت ترتیب ورودی‌ها را به شدت در نظر می‌گیرند، زیرا آن‌ها از توکن‌های ورودی و اطلاعات موقعیتی برای درک معنای جملات استفاده می‌کنند. در وظایف مرتبط مانند تطبیق پرسش و سند، ترتیب ورودی‌ها می‌تواند تأثیر قابل توجهی بر عملکرد مدل داشته باشد.
ترتیب فعلی: این ترتیب معمولاً برای وظایف جستجو و بازیابی اطلاعات منطقی است، زیرا مدل ابتدا پرسش را می‌بیند و سپس سند مرتبط را بررسی می‌کند.
ترتیب معکوس : ممکن است باعث شود مدل بیشتر بر روی سند تمرکز کند و ارتباط آن با پرسش را کمتر در نظر بگیرد.

# **بخش دوم:**

<div style="font-family: Roboto; width: 85%; font-size: 20px;">
در این قسمت از تمرین سعی کردم موتور جستجو گری بسازم که با استفاده از مدل قبلی بتواند ۵ سند اول مرتبط به کوئری مد نظر را باز گرداند.
در این کد بعد از ایمپورت کردن کتابخانه های مورد نیاز مدل و توکنایزر مناسب آن را بارگذاری کرذم و مدل را در حالت ارزیابی قرار دادم تا در طول محاسبات گرادیان ها محاسبه نشوند. سپس تابع get_relevance_score را تعریف کردم که این تابع مسئول محاسبه نمره مرتبط بودن سند با یک کوئری است.
این تابع کوئری، سند، مدل و توکنایزر را دریافت می کند سپس کوئری و سند با استفاده از توکنایزر به قالب ورودی تبدیل می شوندو در مرحله بعد با استفاده از مدل خروجی logits محاسبه می شود و احتمالات با استفاده از تابع softmaxمحسابه شده و احتمال کلاس مثبت به عنوان نمره میزار مرتبط بودن سند با کوئری بازگردانده مب شود و این نمره مقداری بین ۰ و ۱ است.
در مرحله بعد تابع rank_documents_for_query را تعریف کردم که این تابع برای یک کوئری خاص اسناد را بر اساس میزان ارتباط مرتب میکند. این تابع کوئری، سند، مدل، توکنایزر و تعداد اسنادی که میخواعین باز گردانده شود را میگیرد سپس برای هر سند نمره ارتباط محاسبه می شود. در مرحله بعد اسناد بر اساس نمره به ترتیب نزولی مرتب میشوند و در خروجی لیست کامل اسناد مرتب شده به همراه k سند برتر برکردانده می شود. در مرحله آخر برای نمایش هدف سوال یک کوئری به صورت رندم انتخاب شده و ۵ سند برتر برای آن به همراه نمره ارتباط منتشر می شود.


In [21]:
import random


import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import nltk
import numpy as np
from tqdm import tqdm
model_path = "/content/drive/MyDrive/IR5/bert_model_output/checkpoint-993"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()
def get_relevance_score(query, document, model, tokenizer, device="cuda"):
    inputs = tokenizer(
        query,
        document,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        score = probs[0, 1].item()
    return score
def rank_documents_for_query(query, documents, model, tokenizer, top_k=5, device='cpu'):
    scored_docs = []
    for doc in documents:
        score = get_relevance_score(query, doc, model, tokenizer, device=device)
        scored_docs.append((doc, score))

    scored_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)

    full_ranked_docs = scored_docs
    top_docs = scored_docs[:top_k]

    return full_ranked_docs, top_docs

unique_queries = list(set(test_data["query"]))

sample_query = random.choice(unique_queries)
print('sample query:', sample_query)


_, top5 = rank_documents_for_query(sample_query, test_data['document'], model, tokenizer, top_k=5)
print("Top 5 docs for query:", sample_query, "\n")
for i, (doc, score) in enumerate(top5):
    print(f"Rank {i+1}, Score={score:.4f}")
    print("Document (first 150 chars):", doc[:150])
    print("---")

sample query: conflict of interests in italy
Top 5 docs for query: conflict of interests in italy 

Rank 1, Score=0.9437
Document (first 150 chars): An extraordinary political year limps to a close in Italy today amid 
dispiriting national recognition that, after months of great ferment and hope, 

---
Rank 2, Score=0.8846
Document (first 150 chars): China, Britain Near Accord on New Airport: The governments of the two nations, 
which are discussing funding for Hong Kong's new airport, could sign a
---
Rank 3, Score=0.8576
Document (first 150 chars): Britain and France, which contribute the most troops to the U.N. Protection 
Force in Bosnia-Herzegovina, may decide as early as today whether to sold
---
Rank 4, Score=0.8305
Document (first 150 chars): The ghost of Henry Cabot Lodge is stalking President Clinton. Seventy-five years ago, in the aftermath of World War I, Sen. Lodge 
successfully led th
---
Rank 5, Score=0.8185
Document (first 150 chars): Lawsuits Exchanged in Magma Takeove

<div style="font-family: Roboto; width: 85%; font-size: 20px;">
این کد به این منظور نوشته شده است که اسناد مرتبط با یک کوئری خاص را از بین داده های تست استخراج کند و خلاصه ای از هر سند را نمایش دهد. همانطور که مشخص است برای کوئری نمونه در کد بالا فقط یک سند مرتبط وجود دارد که همین سند در رشته کد بالا به عنوان سندی با بیشترین میزان ارتباط مشخص شده . همچنین مدل ۴ سند دیگر نیز بازگردانده است که طبق دیتا ست این اسناد نا مرتبط هستند اما احتمالا به دلیل وجود کلمات مشترک مدل آن ها را مرتبط ارزیابی کرده است.

In [22]:

filtered_data = test_data.filter(lambda x: x["query"] == sample_query)

print(f"Found {len(filtered_data)} documents for query: '{sample_query}'\n")

for i in range(len(filtered_data)):
    doc_text = filtered_data[i]["document"]
    doc_label = filtered_data[i]["labels"]

    snippet = doc_text[:150].replace("\n", " ")

    print(f"Document #{i+1} (Label={doc_label}) - first 150 chars:")
    print(snippet)
    print("---")

Filter:   0%|          | 0/330 [00:00<?, ? examples/s]

Found 1 documents for query: 'conflict of interests in italy'

Document #1 (Label=1) - first 150 chars:
An extraordinary political year limps to a close in Italy today amid  dispiriting national recognition that, after months of great ferment and hope,  
---


<div style="font-family: Roboto; width: 85%; font-size: 20px;">
در این قسمت نیز سعی کردم کدی بنویسم که با فراخوانی مدل روی داده های تست بتوانم معیار های ارزیابی معمول را برای مدل محاسبه کنم.
به همین منظور ابتدا معیار های ارزیابی را برای مدل تعریف کرده ام. معیار precisionبه این صورت محاسبه می شود که لیستی از  لیبل های اسناد بازیابی شده دریافت می کند و میانگین اسناد مرتبط در  بین k سند را منتشر می کنذ.
معیار recall به این صورت تعریف شده که بیستس از برچسب اسناد بازیابی شده را دریافت مند و با توجه به تعداد کل اسناد مرتبط و تعداد کل اسناد بازیابی شده، نسبت اسناد مرتبط بازیابی شده به کل اسناد مرتبط را به دست آورد.
در مرحله بعد معیار average precision تعرف شده که لبستی از لیبل های اسناد بازیابی شده را دریافت میکند و میانگین دقت را در تمام اسناد مرتبط محاسبه می کند و mapنیز این میانگین را برای تمام کوئری ها محاسبه میکند.
همچنین تابع ndcg نیز برای محاسبه اینکه چه درصدی از سند های مرتبط در رتبه های اول قرار گرفته اند لیستی از برچسب های اسناد بازیابی شده را دریافت میکند و مقدار ndcg را برای kسند اول محاسبه میکند.
در مرحله بعد از تابع get_relevance_scoreکه. در رشته کد های قبلی نوشته بودیم استفاده میکنیم تا نمره مرتبط بودن یک کوئری با سند را به دست بیاوریم.
همچنین تابع rank_documents_for_query را نیز مثل رشته کد قبلی برای رتبه بندی اسناد برای یک کوئری بر اساس نمره مرتبط بودن مجدد نوشته ایم.
در ادامه نیز معیار های ارزیابی را برای این مدل محاسبه کرده ایم که این معیار ها به شرح زیر می باشند:
<br>
MAP@5: 0.7207
Recall@10: 0.8660
Precision@5: 0.6029
nDCG@5: 0.7806
<br>
معیار های ارزیابی با روش bm25 نیز به شرح زیر ند:
<br>
MAP@5: 0.399
Recall@10: 0.432
Precision@5: 0.3346
nDCG@5: 0.5335
<br>
همانطور که از مقادیر مشخص است مدل در همه معیار ها عملکرد بهتری نسبت به روش bm25داشته است.
مدل توانسته به طور میانگین برای ۵ سند اول استخراج شده حدودا ۷۲٪ اسناد مرتبط را استخراج کند.
همچنین مدل توانسته است 86% کل اسناد مرتبط را به درستی لیبل گذاری کند. مذل توانسته حدودا۷۸ درصد اسنا مرتبط را در بین ۵ رتبه اول کوئری مربوطه قرار دهذ.


In [10]:
import numpy as np

def precision_at_k(r, k):

    r = np.asarray(r)[:k]
    return np.mean(r)

def recall_at_k(r, k, total_relevant):

    r = np.asarray(r)[:k]
    return np.sum(r) / total_relevant if total_relevant > 0 else 0.0

def average_precision(r, k=None):

    if k is not None:
        r = np.asarray(r)[:k]
    else:
        r = np.asarray(r)
    out = [precision_at_k(r, i + 1) for i in range(len(r)) if r[i]]
    if not out:
        return 0.0
    return np.mean(out)

def mean_average_precision(rs, k=None):
    return np.mean([average_precision(r, k) for r in rs])

def ndcg_at_k(r, k):

    r = np.asarray(r)[:k]
    dcg = np.sum((2**r - 1) / np.log2(np.arange(2, r.size + 2)))

    ideal_r = np.sort(r)[::-1]
    ideal_dcg = np.sum((2**ideal_r - 1) / np.log2(np.arange(2, ideal_r.size + 2)))
    return dcg / ideal_dcg if ideal_dcg > 0 else 0.0
def rank_documents_for_query(query, documents, model, tokenizer, top_k=5, device='cpu'):
    scored_docs = []
    for doc in documents:
        score = get_relevance_score(query, doc, model, tokenizer, device=device)
        scored_docs.append((doc, score))

    scored_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)

    full_ranked_docs = scored_docs
    top_docs = scored_docs[:top_k]

    return full_ranked_docs, top_docs
import torch
from tqdm import tqdm
from collections import defaultdict


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def get_relevance_score(query, document, model, tokenizer, device=device):
    inputs = tokenizer(
        query,
        document,
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        score = probs[0, 1].item()
    return score


unique_queries = list(set(test_data["query"]))


query_docs = defaultdict(list)
query_labels = defaultdict(list)

for q, d, l in zip(test_data["query"], test_data["document"], test_data["labels"]):
    query_docs[q].append(d)
    query_labels[q].append(l)


all_map = []
all_recall = []
all_precision = []
all_ndcg = []


k_map = 5
k_recall = 10
k_precision = 5
k_ndcg = 5


for query in tqdm(unique_queries, desc="Processing Queries"):
    documents = query_docs[query]
    labels = query_labels[query]
    total_relevant = sum(labels)


    ranked_docs, top_docs = rank_documents_for_query(query, documents, model, tokenizer, top_k=k_map, device=device)


    ranked_labels = [labels[i] for i in np.argsort([doc[1] for doc in ranked_docs])[::-1]]


    map_score = average_precision(ranked_labels, k=k_map)
    recall_score = recall_at_k(ranked_labels, k_recall, total_relevant)
    precision_score = precision_at_k(ranked_labels, k_precision)
    ndcg_score = ndcg_at_k(ranked_labels, k_ndcg)


    all_map.append(map_score)
    all_recall.append(recall_score)
    all_precision.append(precision_score)
    all_ndcg.append(ndcg_score)


mean_map = np.mean(all_map)
mean_recall = np.mean(all_recall)
mean_precision = np.mean(all_precision)
mean_ndcg = np.mean(all_ndcg)

print(f"MAP@{k_map}: {mean_map:.4f}")
print(f"Recall@{k_recall}: {mean_recall:.4f}")
print(f"Precision@{k_precision}: {mean_precision:.4f}")
print(f"nDCG@{k_ndcg}: {mean_ndcg:.4f}")


Processing Queries: 100%|██████████| 64/64 [00:05<00:00, 11.21it/s]

MAP@5: 0.7207
Recall@10: 0.8660
Precision@5: 0.6029
nDCG@5: 0.7806


# **بخش سوم:**

In [7]:
#YOUR CODE HERE
!pip install together

In [25]:
print(train_data[2:3])

{'query': ['conflict of interests in italy'], 'document': ['In a heady spring of change and promise, he vaulted to power almost \neffortlessly, bringing new ideas, plans and promises to a country dispirited by \ncronyism and political corruption. Now, seven months later, Silvio Berlusconi\'s \ndays in power seem numbered. One day after magistrates interrogated the prime minister in a bribery case \ninvolving his Fininvest business empire -- "a spectacle of injustice," he \nsnapped -- Berlusconi\'s right-wing coalition splintered in Parliament on \nWednesday as support for his leadership eroded. Today many Berlusconi allies and enemies alike arise convinced that the \ngovernment that promised Italians a brighter future may not live out the year. \n"The Berlusconi chapter is closed. . . . We think the government is finished," \nsaid Marco Formentini, mayor of Milan and a nominal Berlusconi ally. "It\'s time \nto think of a new political alliance." Formentini represents the Northern Leagu

<div style="font-family: Roboto; width: 85%; font-size: 20px;">
در این قسمت از کد تلاش کردم تا با استفاده از مدل های زبانی بزرگ پیش بینی تحتمال ارتباط بین پرس و جوی کاربر و سند را انجام دهم.
به همین منظور و برای آماده سازی داده ها و همچنین برای مرتب سازی آن ها جهت استفاده در به دست آوردن معیار های ارزیابی به شکل زیر عمل کرده ام.
ابتدا لیست data, data2 را ساختم و سپس داده های تست را با لیبلشان در لیست data2 و بدون لیبل در لیست dataقرار دادم همچنین شمانده n را نیز تعریف کردم که با استفاده از آن برای مقایسه پاسخ های ساخته شده توسط llm و دیتا ست تست بتوان این دو دیتا ست را باهم دیگر merge کنم.

In [8]:
data = []
data2=[]
n = 0
for i in range(len(test_data)):
    n +=1
    data.append({
        "number": n,
        "query": test_data["query"][i],
        "document": test_data["document"][i],
    })
    data2.append({
        "number": n,
        "query": test_data["query"][i],
        "document": test_data["document"][i],
        "labels": test_data["labels"][i]
    })
print(data[:3])
print(data2[:1])

[{'number': 1, 'query': 'hunger strikes', 'document': 'The cancellation of the 1994 baseball season has left many advertisers \nscrambling for scarce commercial time on other programs and may lead some firms \nto re-evaluate future baseball-related promotions and ads. "A controversy has been created and it\'s something that these corporations \nwould rather not get involved in," said Alan Friedman, editor of Team Marketing \nReport, a Chicago-based sports business publication. "It might lead some \ncompanies to cut down on advertising because they fear fan reaction next year." Baseball owners wrote off the rest of the regular season as well as the \nplayoff games and World Series in October after failing to reach an agreement \nto end a 34-day players strike. The cancellation comes at a time when many consumer product companies -- such \nas beer, automobile and movie makers -- begin their seasonal fall ad campaigns \nand introduce new products. Such companies spend about $500 million a

In [10]:
print(len(data2))

331


<div style="font-family: Roboto; width: 85%; font-size: 20px;">
در ادامه جهت استفاده از مدل زبانی llama 3.1 api keyخود در سایت together ai را وارد کرده ام و مدل زبانی را تعریف کردم همچنین به عنوان پرامپت اول به مدل القا کردم که او یک موتور جست و جو گر پیشرفته است و از او درخواست کردم با توجه به مرتبط بودن یا نبودن هر سند با کوئری مروبطه فقط مقدار ۰ یا ۱ بازگرداند و این کار را برای دیتا ست به ترتیب انجام دهد. در ادامه باتوجه به داده ها در دیتا ست تست دو نمونه را به عنوان اسناد مرتبط و نامرتبط برای یک کوئری مشخص را به عنوان مثال به مدل دادم و در مرحله بعد داده ها همراه پرامپت را برای مدل ارسال کردم و شماره کوئری، کوئری، سند، لیبل مربوطه را در فایل csv ذخیره کردم.

In [13]:
import os
os.environ["TOGETHER_API_KEY"] = os.getenv("TOGETHER_API_KEY", "")
from together import Together
import pandas as pd

# Step 01: Verify Together API Key is set
api_token = os.getenv("TOGETHER_API_KEY")
if not api_token:
    raise ValueError("API token not found. Set TOGETHER_API_KEY environment variable.")

# Step 02: Define model and task
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo"  # Replace with the specific model name for classification if needed
task_description = """
You are an advanced search engine. Given a user query and a list of documents, determine the relevance of each document to the query.
For each query-document pair, respond with '1' if the document is relevant and '0' if it is not, so just give '0' or '1' and do it respectively.
""" # Replace with your Task Description
together_client = Together()

few_shot_examples = """
### Example 1
**Number:** 1
**Query:** u.n. us invasion of haiti
**Document 1:** 'President Jean-Bertrand Aristide of Haiti addresses the U.N. General Assembly \ntoday and seems certain to attract one of the most attentive audiences of the \nsession. More than a dozen presidents, prime ministers and foreign ministers address \nthe assembly every day during the annual general debate in a sometimes \nstupefying deluge of words.But everyone wants to hear what Aristide plans for \nhis country when he returns there sometime later this month. Aristide has spoken to the General Assembly twice before. In 1991, he \naddressed the general debate only a few days before a military coup ousted him \nfrom office. Although in exile in 1992, he addressed the General Assembly again \nas the U.N.-recognized Haitian leader. The Security Council has approved \nsending a U.N. peacekeeping force of 6,000 troops into Haiti next year to \nrelieve the U.S. troops in the island nation now.
**Relevance Label:** 1

**Document 2:** In a heady spring of change and promise, he vaulted to power almost \neffortlessly, bringing new ideas, plans and promises to a country dispirited by \ncronyism and political corruption. Now, seven months later, Silvio Berlusconi\'s \ndays in power seem numbered. One day after magistrates interrogated the prime minister in a bribery case \ninvolving his Fininvest business empire -- "a spectacle of injustice," he \nsnapped -- Berlusconi\'s right-wing coalition splintered in Parliament on \nWednesday as support for his leadership eroded. Today many Berlusconi allies and enemies alike arise convinced that the \ngovernment that promised Italians a brighter future may not live out the year. \n"The Berlusconi chapter is closed. . . . We think the government is finished," \nsaid Marco Formentini, mayor of Milan and a nominal Berlusconi ally. "It\'s time \nto think of a new political alliance." Formentini represents the Northern League, a populist-federalist party that \nhad been a pillar of Berlusconi\'s coalition. On Wednesday, the League broke \nwith the government, voting with the opposition for formation of a special \ncommission to oversee a reorganization of Italian broadcasting. Berlusconi\'s \n$7-billion-a-year interests include ownership of three of Italy\'s seven \nnational television channels. League supporters exchanged shouts and abuse Wednesday on the Parliament floor \nwith legislators of Berlusconi\'s Forza Italia and the National Alliance, its \nextreme-right-wing stablemate. "The moment of truth is coming," said Alliance \nleader Gianfranco Fini. An atmosphere of political crisis hung over chilly, polluted streets around \nthe downtown Parliament on Wednesday night as politicians maneuvered in search \nof a fulcrum. In the next few days, the Senate is likely to approve Berlusconi\'s austerity \n1995 budget; it must pass by year\'s end. Once that is done, Berlusconi says he \nwill ask for a vote of confidence, a move that could bring down his government. The 58-year-old Milan entrepreneur emerged from an eight-hour session with \nmagistrates in Milan on Tuesday in a cold fury. The magistrates believe he may \nhave known of about $205,000 in bribes allegedly paid by Fininvest to tax \ninspectors in 1990 and 1991. "There is no shred of paper or any person to affirm that I ordered any illegal \nacts or was aware of any in conducting my businesses," Berlusconi said. Claiming their inquiries are politically motivated, the government is dueling \nwith judges who have led a corruption probe that has ensnared more than 3,000 \nbusiness leaders, public officials and politicians. "Against me and my government there is a plot whose vastness and intensity \ncould be compared to a coup d\'etat," Berlusconi complained after meeting the \njudges. Conspirators, he said, "want a crisis, and they are working so that \nthings go badly with the sole purpose of getting rid of me.
**Relevance Label:** 0

"""
# Step 03: Input data
data = data

# Step 04: Function to perform classification
def classify_relevance(pairs, model_name):
    results = []
    for pair in pairs:
        # Prepare the input for the model
        prompt = f"{task_description}\n\nNumber: {pair['number']}\n\nQuery: {pair['query']}\nDocument: {pair['document']}\n\nRelevance Label:" # Replace with your Prompt Template
        print(prompt)
        try:
            # Query Together API
            response = together_client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=1,  # Limit tokens to encourage concise output
            )
            print(response.choices[0].message.content)
            label = response.choices[0].message.content
            # Ensure label is binary (0 or 1), fallback to 'Error' otherwise
            label = "1" if label == "1" else "0" if label == "0" else "Error"
            results.append({
                "Number": pair["number"],
                "Query": pair["query"],
                "Document": pair["document"],
                "Relevance Label": label,
            })
        except Exception as e:
            print(f"Error processing pair: {pair}. Error: {e}")
            results.append({
                "Number": pair["number"],
                "Query": pair["query"],
                "Document": pair["document"],
                "Relevance Label": "Error",
            })
        print("\n---------------------------------------\n")
    return results

# Step 05: Run the classification
results = classify_relevance(data, model_name)

# Step 06: Save results to a CSV file
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo"

results = classify_relevance(data, model_name)

results_df = pd.DataFrame(results)

output_file = "/content/drive/MyDrive/IR5/relevance_classification_results.csv"
results_df.to_csv(output_file, index=False)
print(f"Results saved to {output_file}")


Streaming output truncated to the last 5000 lines.
1

---------------------------------------


You are an advanced search engine. Given a user query and a list of documents, determine the relevance of each document to the query.
For each query-document pair, respond with '1' if the document is relevant and '0' if it is not, so just give '0' or '1' and do it respectively.


Number: 249

Query: hunger strikes
Document: I've been working in the Diocese of San Cristobal de las Casas for four years. 
We hear a lot of criticism of the church in Chiapas. We've been blamed by the 
cattle barons and other large landowners, and by the state government for 
whatever conflict arises; we seem to be their scapegoat. What has happened is that in the church we've taken seriously the documents of 
the Second Vatican Council and the documents produced by the bishops of Latin 
America. Around 1975, with the leadership of Bishop Samuel Ruiz Garcia, the 
church in Chiapas made a radical change, deciding t

In [14]:
!pip install pandas scikit-learn

<div style="font-family: Roboto; width: 85%; font-size: 20px;">
در این قسمت از کد نیز دو دیتا ست اصلی تست و پیش بینی های مدل رو به دیتا فریم پانداس تبدیل کزدم و دو دیتا ست را باهم mergeکردم تا بتوانم لیبل های اصلی و پیش بینی شده را در کنار هم قرار دهم. فقط مشکلی که تو این قسمت برام پیش اومد این بود که llm برای قسمت از داده ها لیبلی پیش بینی نمیکرد و صرفا یکسری توکن و متن تولید میکرد و وسط دیتا ست پیش بینی ها قرار میداد به همین منظور من اون یکسری داده رو مجبور شدم به صورت دستی پاک کنم البته این ها سطر های مشخصی بودند که به پیوست تمرین براتون ارسال میکنم که دو دیتا ست رو مشاهده بفرمایید.

In [40]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
test_df = pd.DataFrame(data2)
predictions_df = pd.read_csv('/content/drive/MyDrive/IR5/relevance_classification_results2.csv')

merged_df = pd.merge(test_df, predictions_df, on='number', how='inner')

print(merged_df.head())

   number                  query_x  \
0       1           hunger strikes   
1       2    conflict in palestine   
2       3  death of nirvana leader   
3       4      g7 summit in naples   
4       5            mobile phones   

                                          document_x  labels  \
0  The cancellation of the 1994 baseball season h...       0   
1  We're in agreement with Mr. Hall's (TV Times, ...       0   
2  Kurt Cobain's death isn't exactly like Victor ...       1   
3  It's been ages since anyone of Cash's stature ...       0   
4  President Clinton's beleaguered effort to over...       0   

                   query_y                                         document_y  \
0           hunger strikes  The cancellation of the 1994 baseball season h...   
1    conflict in palestine  We're in agreement with Mr. Hall's (TV Times, ...   
2  death of nirvana leader  Kurt Cobain's death isn't exactly like Victor ...   
3      g7 summit in naples  It's been ages since anyone of Cas

<div style="font-family: Roboto; width: 85%; font-size: 20px;">
در این قسمت هم ستون لیبل مربوط به داده های تست و پیش بینی ها را جدا کردم و با استفاده از کتابخانه skitlrearnمقادیر precision, recall, f1 , accuracy رو محاسبه کردم که این مفادیر به شرح زیرند:
<br>
Precision: 0.9854
Recall: 0.7941
F1 Score: 0.8795
Accuracy: 0.8882
<br>
همچنین مقدایر مدل bertنیز به شرح زیر می باشند:
<br>
eval_accuracy: 0.8606
eval_precision: 0.8098
eval_recall: 0.9313
eval_f1: 0.8663
<br>
مقدار معیار precision در حالت استفاده از llm برابر ۹۸ درصد بوده که بیشتر از مدل برت می باشد و به این معناست مدل تواسنته است اسناد مرتبط را در بین همه اسناد بهتر تشخیص دهد نسبت به مدل برت. اما مقدار recall برای این مدل برابر ۷۹ درصد بوده که از مدل برت خیلی کمتر است و به این معناست که مدل نتوانسته درصد بالایی از سند های مرتبط را به درستی شناسایی کند بر عکس مدل برت. همچنین مقدار f1 scoreکه میانگینی وزن دار بین دقت و فراخوانی است در مدل دوم که از llm استفاده شده بیشتر است و نشان از عملرد بهتر این مدل در حالت کلی دارد. میزان accuracy هم در دو مدل اختلاف کمی دارد که مدل با استفاده از llm از نظر این معیار عملکرد بهتری داشته است.

In [43]:

y_true = merged_df['labels']
y_pred = merged_df['label']

precision = precision_score(y_true, y_pred)


recall = recall_score(y_true, y_pred)

f1 = f1_score(y_true, y_pred)

accuracy = accuracy_score(y_true, y_pred)



print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'Accuracy: {accuracy:.4f}')

Precision: 0.9854
Recall: 0.7941
F1 Score: 0.8795
Accuracy: 0.8882


# **نکات مهم**

<h4 dir="rtl" style="font-family: Vazir; width: 85%;">فایل ارسالی شما باید با فرمت زیر نامگذاری شود: <code>IIR-CA5-STUDENTID.ipynb</code></h4>

<h4 dir="rtl" style="font-family: Vazir; width: 85%;">نحوه انجام این تمرین:</h4>
<ul dir="rtl" style="font-family: Vazir; width: 85%; font-size: 16px;">
<li> کدها و تفسیرهای هربخش را به طور مشخص در همین نوت‌بوک بنویسید. سعی کنید هربخش به طور مشخصی جداشده باشد و ساختار نوت‌بوک خوانا باشد.</li>
</ul>

<h4 dir="rtl" style="font-family: Vazir; width: 85%;">صداقت علمی:</h4> <ul dir="rtl" style="font-family: Vazir; width: 85%; font-size: 16px;"> <li>ما نوت‌بوک‌های تعداد مشخصی از دانشجویان که به صورت تصادفی انتخاب می‌شوند، بررسی خواهیم کرد. این بررسی‌ها اطمینان حاصل می‌کنند که کدی که نوشتید واقعاً پاسخ‌های موجود در نوت‌بوک شما را تولید می‌کند. اگر پاسخ‌های صحیح را در نوت‌بوک خود بدون کدی که واقعاً آن پاسخ‌ها را تولید کند تحویل دهید، این یک مورد جدی از عدم صداقت علمی محسوب می‌شود.</li> <li>ما همچنین بررسی‌های خودکاری را برای تشخیص سرقت علمی در نوت‌بوک‌های کولب انجام خواهیم داد. کپی کردن کد از دیگران نیز یک مورد جدی از عدم صداقت علمی محسوب می‌شود.</li> </ul>

<h4 dir="rtl" style="font-family: Vazir; width: 85%;">توضیحات تکمیلی:</h4> <ul dir="rtl" style="font-family: Vazir; width: 85%; font-size: 16px;">
<li>
خوانایی و دقت بررسی‌ها در گزارش نهایی از اهمیت ویژه‌ای برخوردار است. به تمرین‌هایی که به صورت کاغذی تحویل داده شوند یا به صورت عکس در سایت بارگذاری شوند، ترتیب اثری داده نخواهد شد.</li>
<li>
 همه‌ی کدهای پیوست گزارش بایستی قابلیت اجرای مجدد داشته باشند. در صورتی که برای اجرای مجدد آن‌ها نیاز به تنظیمات خاصی می‌باشد، بایستی تنظیمات مورد نیاز را نیز در گزارش خود ذکر کنید.  دقت کنید که  تمامی کدها باید توسط شما اجرا شده باشند و نتایج اجرا در فایل کدهای ارسالی مشخص باشد. به کدهایی که نتایج اجرای آن‌ها در فایل ارسالی مشخص نباشد نمره‌ای تعلق نمی‌گیرد.
</li>
<li>
تمرین تا یک هفته بعد از مهلت تعیین شده با تاخیر تحویل گرفته می‌شود. دقت کنید که شما جمعاً برای تمام تکالیف، ۱۴ روز زمان تحویل بدون جریمه دارید که تنها از ۷ روز آن برای هر تمرین می‌توانید استفاده کنید. در صورتی که این ۱۴ روز به اتمام رسیده باشد، به ازای هر روز تاخیر ده درصد جریمه می‌شود.
</li>
<li>توجه کنید این تمرین باید به صورت تک‌نفره انجام شود و پاسخ‌های ارائه شده باید نتیجه فعالیت فرد نویسنده باشد (همفکری و به اتفاق هم نوشتن تمرین نیز ممنوع است). در صورت مشاهده
 تشابه به همه افراد مشارکت‌کننده، نمره تمرین صفر و به استاد گزارش می‌گردد.
 </li>
 <li>برای مطالعه بیشتر درباره‌ی فرمت مارک‌داون می‌توانید از <a href="https://github.com/tajaddini/Persian-Markdown/blob/master/learn-MD.md">این لینک</a> مطالعه کنید.
 </li>
 <li>در صورت بروز هرگونه مشکل، می‌توانید با ایمیل
<code>mr.alaei@ut.ac.ir</code>
در ارتباط باشید.
 </li>
 </ul>
 </div>

<div dir="rtl" style="font-family: Vazir; width: 85%; font-size: 16px;">
    <p><strong>مهلت تحویل بدون جریمه:</strong> ۳۰ آذر ۱۴۰۳</p>
    <p><strong>مهلت تحویل با تاخیر (با احتساب جریمه):</strong> ۷ دی ۱۴۰۳</p>
</div>